# 01 · Clean the hospital data (Kaggle: Diabetes 130-US Hospitals, 1999–2008)

**Question for the project:** which diabetic patients did the old standard of care fail, and has today's market caught up for them?

This notebook turns the raw Kaggle file into one clean row per patient. Every cleaning decision is written down, with the row count after each step, because those decisions change the answer.

**Source:** [Diabetes 130-US Hospitals for Years 1999–2008](https://www.kaggle.com/datasets/brandao/diabetes) (Strack et al., 2014; de-identified extract of Cerner's Health Facts database). 101,766 hospital stays of diabetic patients, 1–14 days long, with lab tests and medications recorded.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable when run from notebooks/

import numpy as np
import pandas as pd
import duckdb
import kagglehub
from src.config import PROCESSED, update_results

path = Path(kagglehub.dataset_download("brandao/diabetes")) / "diabetic_data.csv"
raw = pd.read_csv(path, na_values="?", low_memory=False)
print(raw.shape)
raw.head(3)

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## 0. What one row looks like, and a privacy check

Each row is **one hospital stay** of a diabetic patient. Before cleaning anything I check what's in the file: 50 columns, grouped below, and whether any of them could identify a person.

In [2]:
GROUPS = {
    "Record codes": ["encounter_id", "patient_nbr"],
    "Demographics": ["race", "gender", "age", "weight"],
    "The stay": ["admission_type_id", "discharge_disposition_id", "admission_source_id", "time_in_hospital",
                 "payer_code", "medical_specialty", "num_lab_procedures", "num_procedures", "num_medications"],
    "Prior year": ["number_outpatient", "number_emergency", "number_inpatient"],
    "Diagnoses": ["diag_1", "diag_2", "diag_3", "number_diagnoses"],
    "Lab results": ["max_glu_serum", "A1Cresult"],
    "Diabetes medications (23)": [c for c in raw.columns[24:47]],
    "Medication summary": ["change", "diabetesMed"],
    "Outcome": ["readmitted"],
}
assert sum(len(v) for v in GROUPS.values()) == raw.shape[1]

SHOW = ["race", "gender", "age", "admission_type_id", "time_in_hospital", "num_medications", "number_inpatient",
        "diag_1", "A1Cresult", "metformin", "insulin", "change", "readmitted"]
sample = raw.sample(6, random_state=7)[SHOW].fillna("?")
sample

,race,gender,age,admission_type_id,time_in_hospital,num_medications,number_inpatient,diag_1,A1Cresult,metformin,insulin,change,readmitted
91725,AfricanAmerican,Female,[50-60),3,1,12,0,996,?,No,No,No,NO
76741,Caucasian,Male,[80-90),2,4,23,1,428,>7,No,Up,Ch,NO
11792,Caucasian,Male,[60-70),1,6,13,0,250.02,>8,No,No,Ch,>30
58861,Caucasian,Female,[30-40),2,4,14,0,641,?,Steady,Steady,Ch,>30
63383,AfricanAmerican,Female,[50-60),2,7,28,0,250.7,?,Steady,Up,Ch,NO
58823,Caucasian,Female,[90-100),1,2,11,0,250.8,?,No,No,No,>30


In [3]:
# Privacy check: could any column identify a person?
date_like = [c for c in raw.columns if raw[c].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}").any()]
free_text = [c for c in raw.select_dtypes("object") if raw[c].nunique() > 1000]
privacy_kaggle = {
    "columns": int(raw.shape[1]),
    "date_columns": date_like,                                  # no admission or birth dates
    "free_text_columns": free_text,                             # diagnosis codes are the only high-cardinality text
    "age_values": sorted(raw.age.unique()),                     # 10-year bands only
    "name_address_zip_columns": [c for c in raw.columns if any(k in c.lower() for k in ["name", "address", "zip", "ssn", "phone", "dob", "birth"])],
    "patient_code_note": "patient_nbr is a de-identified code assigned by the data provider, not a medical record number",
}
privacy_kaggle

{'columns': 50,
 'date_columns': [],
 'free_text_columns': [],
 'age_values': ['[0-10)',
  '[10-20)',
  '[20-30)',
  '[30-40)',
  '[40-50)',
  '[50-60)',
  '[60-70)',
  '[70-80)',
  '[80-90)',
  '[90-100)'],
 'name_address_zip_columns': [],
 'patient_code_note': 'patient_nbr is a de-identified code assigned by the data provider, not a medical record number'}

**No names, dates, addresses or ZIP codes; age only in 10-year bands.** The two ID columns are codes the data provider assigned during de-identification. The only high-cardinality text columns are the ICD-9 diagnosis codes. This is the standard published research extract (Strack et al., 2014), released for public use.

## 1. What's missing

`?` marks a missing value in this file. Three columns are mostly empty, so I decide what to do with each one before modeling.

In [4]:
missing = (raw.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

weight               96.9
max_glu_serum        94.7
A1Cresult            83.3
medical_specialty    49.1
payer_code           39.6
race                  2.2
diag_3                1.4
diag_2                0.4
dtype: float64

- **weight** is 97% missing → drop it.
- **payer_code** (insurance) and **medical_specialty** (the admitting doctor's specialty) are 40–49% missing. The missingness itself may carry information (for example, an ER admission with no specialty recorded), so I keep both and label the gap `Unknown` instead of dropping rows.
- **A1C result** and **max glucose serum** are mostly missing because the test **was not done**. That's a finding, not a data problem, so `None` becomes `Not tested`.

## 2. Cleaning steps

| Step | Why |
|---|---|
| Remove stays that ended in death or hospice | Those patients can't be readmitted, so they would make readmission look rarer than it is. |
| Keep each patient's **first** stay only | 71,518 patients have 101,766 stays. Repeat patients are counted many times and are more likely to be readmitted, which would leak information across the train/test split. |
| Target = readmitted **within 30 days** | 30-day readmission is the standard quality measure (Medicare's Hospital Readmissions Reduction Program penalizes it). |

In [5]:
log = [("Raw hospital stays", len(raw))]

# Discharge codes 11, 19, 20, 21 = expired; 13, 14 = hospice (from the dataset's ID mapping file)
DEATH_HOSPICE = [11, 13, 14, 19, 20, 21]
df = raw[~raw.discharge_disposition_id.isin(DEATH_HOSPICE)].copy()
log.append(("Remove death / hospice discharges", len(df)))

readmit_all_stays = (df.readmitted == "<30").mean()     # every stay, before keeping one per patient (benchmark in notebook 06)
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
log.append(("Keep first stay per patient", len(df)))

df = df[df.gender != "Unknown/Invalid"]
log.append(("Drop unknown gender", len(df)))

df["readmit_30"] = (df.readmitted == "<30").astype(int)
pd.DataFrame(log, columns=["step", "rows"])

,step,rows
0,Raw hospital stays,101766
1,Remove death / hospice discharges,99343
2,Keep first stay per patient,69990
3,Drop unknown gender,69987


## 3. Group the diagnosis codes

The primary diagnosis (`diag_1`) has about 700 distinct ICD-9 codes. I group them into the 9 categories from the original paper (Strack et al., 2014) so they're usable as features and readable on a slide.

In [6]:
def icd9_group(code):
    if pd.isna(code):
        return "Other"
    if code.startswith(("V", "E")):
        return "Other"
    x = float(code)
    if 390 <= x <= 459 or int(x) == 785: return "Circulatory"
    if 460 <= x <= 519 or int(x) == 786: return "Respiratory"
    if 520 <= x <= 579 or int(x) == 787: return "Digestive"
    if int(x) == 250:                    return "Diabetes"
    if 800 <= x <= 999:                  return "Injury"
    if 710 <= x <= 739:                  return "Musculoskeletal"
    if 580 <= x <= 629 or int(x) == 788: return "Genitourinary"
    if 140 <= x <= 239:                  return "Neoplasms"
    return "Other"

for c in ["diag_1", "diag_2", "diag_3"]:
    df[c + "_group"] = df[c].map(icd9_group)

# Any of the three diagnoses circulatory or kidney-related: the patients today's guidelines single out
df["has_heart_dx"] = df[["diag_1", "diag_2", "diag_3"]].apply(
    lambda r: any(icd9_group(c) == "Circulatory" for c in r), axis=1).astype(int)
KIDNEY = ("403", "404", "584", "585", "586", "250.4")   # hypertensive kidney disease, kidney failure, diabetic kidney disease
df["has_kidney_dx"] = df[["diag_1", "diag_2", "diag_3"]].apply(
    lambda r: any(isinstance(c, str) and c.startswith(KIDNEY) for c in r), axis=1).astype(int)
df.diag_1_group.value_counts()

diag_1_group
Circulatory        21389
Other              12134
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64

## 4. Readable categories

The admission, discharge and source columns are numeric IDs. I map them to labels using the dataset's `IDs_mapping` file, grouping rare codes.

In [7]:
ADMIT_TYPE = {1: "Emergency", 2: "Urgent", 3: "Elective", 7: "Emergency"}
DISCHARGE = {1: "Home", 6: "Home with home health", 8: "Home with home health",
             3: "Skilled nursing / care facility", 4: "Skilled nursing / care facility",
             5: "Skilled nursing / care facility", 22: "Skilled nursing / care facility",
             23: "Skilled nursing / care facility", 24: "Skilled nursing / care facility",
             2: "Transferred to another hospital", 7: "Left against medical advice"}
SOURCE = {7: "Emergency room", 1: "Physician / clinic referral", 2: "Physician / clinic referral",
          3: "Physician / clinic referral", 4: "Transfer", 5: "Transfer", 6: "Transfer"}

df["admission_type"] = df.admission_type_id.map(ADMIT_TYPE).fillna("Unknown")
df["discharge_to"] = df.discharge_disposition_id.map(DISCHARGE).fillna("Other / unknown")
df["admission_source"] = df.admission_source_id.map(SOURCE).fillna("Other / unknown")

df["age_mid"] = df.age.str.extract(r"\[(\d+)-").astype(int) + 5   # "[70-80)" -> 75
df["age_band"] = pd.cut(df.age_mid, [0, 40, 50, 60, 70, 80, 100],
                        labels=["<40", "40-49", "50-59", "60-69", "70-79", "80+"])
df["race"] = df.race.fillna("Unknown")
df["payer_code"] = df.payer_code.fillna("Unknown")
df["A1Cresult"] = df.A1Cresult.fillna("Not tested")
df["max_glu_serum"] = df.max_glu_serum.fillna("Not tested")

# Top specialties kept, the long tail grouped
spec = df.medical_specialty.fillna("Unknown")
spec = spec.replace({"Family/GeneralPractice": "Family / general practice", "InternalMedicine": "Internal medicine",
                     "Emergency/Trauma": "Emergency / trauma", "Surgery-General": "Surgery", "Cardiology": "Cardiology",
                     "Nephrology": "Nephrology", "Orthopedics": "Orthopedics", "Orthopedics-Reconstructive": "Orthopedics",
                     "Radiologist": "Radiology", "Pulmonology": "Pulmonology", "Psychiatry": "Psychiatry",
                     "Endocrinology": "Endocrinology", "ObstetricsandGynecology": "Obstetrics / gynecology"})
spec[spec.str.startswith("Surgery")] = "Surgery"
keep = spec.value_counts().index[:12]
df["specialty"] = spec.where(spec.isin(keep), "Other specialty")
df.specialty.value_counts()

specialty
Unknown                      33652
Internal medicine            10641
Family / general practice     4978
Emergency / trauma            4393
Cardiology                    4207
Surgery                       3685
Other specialty               2763
Orthopedics                   2169
Radiology                      859
Nephrology                     797
Pulmonology                    637
Psychiatry                     613
Obstetrics / gynecology        593
Name: count, dtype: int64

## 5. Medication columns

The 23 drug columns say whether each drug was given during the stay and whether the dose went **Up**, **Down** or stayed **Steady**. Two columns (`examide`, `citoglipton`) are "No" for every patient, so they carry no information and are dropped. The class grouping (biguanide, sulfonylurea, insulin…) is added in notebook 02 from the NLM RxClass API, so it comes from a drug database rather than my own labels.

In [8]:
DRUGS = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", "acetohexamide",
         "glipizide", "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose", "miglitol",
         "troglitazone", "tolazamide", "insulin", "glyburide-metformin", "glipizide-metformin",
         "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"]
constant = [c for c in ["examide", "citoglipton"] if df[c].nunique() == 1]
print("Dropped constant drug columns:", constant)
df = df.drop(columns=constant + ["weight"])

use = pd.DataFrame({d: (df[d] != "No").mean() * 100 for d in DRUGS}, index=["% of patients"]).T
use.sort_values("% of patients", ascending=False).round(1).head(10)

Dropped constant drug columns: ['examide', 'citoglipton']


,% of patients
insulin,51.0
metformin,21.3
glipizide,12.9
glyburide,11.1
pioglitazone,7.5
rosiglitazone,6.7
glimepiride,5.3
repaglinide,1.3
glyburide-metformin,0.7
nateglinide,0.7


## 6. A first look in SQL

Same numbers as pandas, written as SQL, the way I'd run them against a claims database.

In [9]:
con = duckdb.connect()
con.register("patients", df)
con.sql('''
    SELECT
        COUNT(*)                                         AS patients,
        ROUND(100 * AVG(readmit_30), 1)                  AS pct_readmitted_30d,
        ROUND(100 * AVG(CAST(A1Cresult <> 'Not tested' AS INT)), 1)   AS pct_a1c_tested,
        ROUND(100 * AVG(CAST(insulin <> 'No' AS INT)), 1)             AS pct_on_insulin,
        ROUND(100 * AVG(CAST("change" = 'Ch' AS INT)), 1)             AS pct_meds_changed,
        ROUND(AVG(time_in_hospital), 1)                  AS avg_days_in_hospital
    FROM patients
''').df()

,patients,pct_readmitted_30d,pct_a1c_tested,pct_on_insulin,pct_meds_changed,avg_days_in_hospital
0,69987,9.0,18.4,51.0,45.0,4.3


In [10]:
out = PROCESSED / "patients.parquet"
df.to_parquet(out, index=False)
summary = con.execute('''SELECT COUNT(*) n, AVG(readmit_30) r, AVG(CAST(A1Cresult <> 'Not tested' AS INT)) a1c FROM patients''').fetchone()
update_results("kaggle_sample", {
    "rows": sample.astype(str).to_dict("records"), "columns_shown": SHOW,
    "groups": {k: v for k, v in GROUPS.items()}, "shape": list(raw.shape), "privacy": privacy_kaggle,
})
update_results("cleaning", {
    "steps": [{"step": s, "rows": int(n)} for s, n in log],
    "patients": int(summary[0]), "readmit_rate": round(summary[1], 4), "a1c_tested_rate": round(summary[2], 4),
    "raw_patients": int(raw.patient_nbr.nunique()), "readmit_rate_all_stays": round(readmit_all_stays, 4),
})
print("saved", out, df.shape)

saved /Users/avijay/Documents/personal-project/data/processed/patients.parquet (69987, 59)
